# BL Travel Companion | Ollama and Pinggy

Adapted from the instructor's W4 Ollama and Pinggy server notebook. Open this copy in Colab, select an available free GPU under **Runtime > Change runtime type**, and run the cells interactively. If Colab denies a GPU or ends the runtime, check its availability before changing where the notebook runs.

Keep this notebook open while running the course exercise. Keep terminal output and the generated tunnel URL out of shared or exported copies.

## Install Ollama

In [ ]:
%%bash
set -euo pipefail
nvidia-smi -L
if ! command -v ollama >/dev/null; then
    apt-get update -qq
    apt-get install -y -qq zstd
    curl -fsSL https://ollama.com/install.sh | sh
fi
ollama --version

## Start Ollama and wait for readiness

In [ ]:
import os
import subprocess
import time

import requests

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
local_url = "http://127.0.0.1:11434"

def ollama_ready():
    try:
        response = requests.get(local_url + "/api/tags", timeout=2)
        return response.ok and isinstance(response.json().get("models"), list)
    except (requests.RequestException, ValueError, AttributeError):
        return False

if not ollama_ready():
    process = globals().get("ollama_process")
    if process is None or process.poll() is not None:
        with open("/tmp/ollama.log", "a") as log:
            ollama_process = subprocess.Popen(
                ["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT
            )

for _ in range(45):
    if ollama_ready():
        break
    time.sleep(1)
else:
    raise RuntimeError("Ollama did not become ready. Inspect /tmp/ollama.log before retrying.")

print("Ollama is ready on the runtime.")

## Download and confirm both models

In [ ]:
for model in ("embeddinggemma", "qwen3:4b"):
    subprocess.run(["ollama", "pull", model], check=True, timeout=1800)

tags_response = requests.get(local_url + "/api/tags", timeout=15)
tags_response.raise_for_status()
tags = {item["name"] for item in tags_response.json()["models"]}
if not tags.intersection({"embeddinggemma", "embeddinggemma:latest"}) or "qwen3:4b" not in tags:
    raise RuntimeError("Expected both model tags. Check the pull output.")
print("Both required models are available.")

## Verify embedding and final text locally

In [ ]:
import math

embedding = requests.post(
    local_url + "/api/embed",
    json={"model": "embeddinggemma", "input": "Travel assistant service check"},
    timeout=(10, 120),
)
embedding.raise_for_status()
vectors = embedding.json().get("embeddings")
if not isinstance(vectors, list) or len(vectors) != 1:
    raise RuntimeError("Expected one embedding vector.")
vector = vectors[0]
if not isinstance(vector, list) or not vector or not all(
    type(value) in (int, float) and math.isfinite(value) for value in vector
):
    raise RuntimeError("Expected a non-empty, finite numeric vector.")
print("Local embedding passed.")

generation = requests.post(
    local_url + "/api/generate",
    json={
        "model": "qwen3:4b",
        "prompt": "Reply with a brief greeting.",
        "stream": False,
        "think": False,
        "options": {"num_predict": 32},
    },
    timeout=(10, 180),
)
generation.raise_for_status()
result = generation.json()
answer = result.get("response")
if result.get("done") is not True or not isinstance(answer, str) or not answer.strip():
    raise RuntimeError("Expected a completed response with non-empty final text.")
print("Local final-text generation passed.")

## Open the Pinggy tunnel for the connection check

After both local checks pass, run the tunnel cell. It uses the lab's reverse SSH forward to port 11434 and rewrites the Host header for Ollama. It runs without an interactive terminal and checks the expected RSA fingerprint in the [Pinggy documentation](https://pinggy.io/help/).

Right-click **Copy the private HTTPS tunnel address**, choose **Copy link address**, and paste the address directly into the project's ignored `.env` as `OLLAMA_BASE_URL`. Anyone with the URL can access the forwarded Ollama API. Keep the notebook session connected during the check and keep the address private. If the tunnel expires, run the tunnel cell again and replace the URL.

In [ ]:
import html
import re
import tempfile
from pathlib import Path

from IPython.display import HTML, display

host = "free.pinggy.io"
fingerprint = "SHA256:nFd5rfJMGuZXvfeRzJ/BtT3TfksAxTWMajcrHRcI7AM"

if "pinggy_process" not in globals() or pinggy_process.poll() is not None:
    tunnel_files = tempfile.TemporaryDirectory(prefix="bl-travel-pinggy-")
    log_path = Path(tunnel_files.name) / "ssh.log"
    host_key_check = Path(tunnel_files.name) / "check_host_key"
    # OpenSSH supplies the hostname, key type, fingerprint, and public key.
    host_key_check.write_text(
        '#!/bin/sh\n'
        'if [ "$2" = ssh-rsa ] && [ "$3" = "' + fingerprint + '" ]; then\n'
        '    printf "%s %s %s\\n" "$1" "$2" "$4"\n'
        'fi\n',
        encoding="utf-8",
    )
    host_key_check.chmod(0o700)

    with log_path.open("wb") as log:
        pinggy_process = subprocess.Popen(
            [
                "ssh", "-T", "-n", "-p", "443",
                "-o", "BatchMode=yes",
                "-o", "StrictHostKeyChecking=yes",
                "-o", "UserKnownHostsFile=/dev/null",
                "-o", "GlobalKnownHostsFile=/dev/null",
                "-o", f"KnownHostsCommand={host_key_check} %H %t %f %K",
                "-o", "ConnectTimeout=15",
                "-o", "ExitOnForwardFailure=yes",
                "-R0:localhost:11434", "qr@" + host, "u:Host:localhost:11434",
            ],
            stdout=log,
            stderr=subprocess.STDOUT,
        )

for _ in range(40):
    output = log_path.read_text(encoding="utf-8", errors="replace")
    address = re.search(r"https://[a-z0-9.-]+\.pinggy(?:-free)?\.link\b", output, re.I)
    if pinggy_process.poll() is not None:
        if "Host key verification failed" in output:
            raise RuntimeError("Pinggy host key verification failed. Check the published fingerprint.")
        raise RuntimeError("Pinggy SSH ended. Check the connection from the runtime before retrying.")
    if address:
        link = html.escape(address.group(), quote=True)
        display(HTML(f'<a href="{link}">Copy the private HTTPS tunnel address</a>'))
        break
    time.sleep(1)
else:
    pinggy_process.terminate()
    raise RuntimeError("Pinggy did not return an HTTPS tunnel within 40 seconds.")

## Shut down the tunnel

Run this cell after local verification. Clear notebook outputs before saving or sharing a copy, then select **Runtime > Disconnect and delete runtime**.

In [ ]:
if "pinggy_process" in globals() and pinggy_process.poll() is None:
    pinggy_process.terminate()
    pinggy_process.wait(timeout=10)

if "tunnel_files" in globals():
    tunnel_files.cleanup()

print("Pinggy is stopped.")